# Question 4: Naive Bayes Spam Classification

**Course:** Natural Language Processing (AI357)  
**Institution:** Sardar Vallabhbhai National Institute of Technology (SVNIT), Surat  

---

### Problem Statement:
Design a Naive Bayes classifier to detect whether an email is spam or not on the following data:
`free`, `win`, and `offer` columns represent binary features (1 if the word appears, 0 otherwise).

| Email | free | win | offer | Class |
| :--- | :---: | :---: | :---: | :--- |
| E1 | 1 | 1 | 1 | Spam |
| E2 | 1 | 1 | 0 | Spam |
| E3 | 1 | 0 | 1 | Spam |
| E4 | 0 | 1 | 1 | Spam |
| E5 | 0 | 0 | 1 | NotSpam |
| E6 | 0 | 1 | 0 | NotSpam |
| E7 | 1 | 0 | 0 | NotSpam |

**Task:** What will be your prediction for new email **E8 with features (0, 0, 0)** where features are in the same order as above? Show complete computations.  
**Marks:** $[4 + 2 = 6]$


## 1. Mathematical Formulation

Using Bernoulli Naive Bayes:
$$P(\text{Class} \mid X) \propto P(\text{Class}) \times \prod_{i=1}^{3} P(x_i \mid \text{Class})$$

For binary features $x_i \in \{0, 1\}$:
$$P(x_i = 1 \mid \text{Class}) = \frac{\text{Count}(x_i = 1 \text{ in Class})}{\text{Count}(\text{Class})}$$
$$P(x_i = 0 \mid \text{Class}) = 1 - P(x_i = 1 \mid \text{Class})$$

For the new email $E8 = (\text{free}=0, \text{win}=0, \text{offer}=0)$:
- Score for Spam:
  $$P(\text{Spam} \mid E8) \propto P(\text{Spam}) \times P(\text{free}=0 \mid \text{Spam}) \times P(\text{win}=0 \mid \text{Spam}) \times P(\text{offer}=0 \mid \text{Spam})$$
- Score for NotSpam:
  $$P(\text{NotSpam} \mid E8) \propto P(\text{NotSpam}) \times P(\text{free}=0 \mid \text{NotSpam}) \times P(\text{win}=0 \mid \text{NotSpam}) \times P(\text{offer}=0 \mid \text{NotSpam})$$


In [1]:
import pandas as pd

training_data = [
    {"Email": "E1", "free": 1, "win": 1, "offer": 1, "Class": "Spam"},
    {"Email": "E2", "free": 1, "win": 1, "offer": 0, "Class": "Spam"},
    {"Email": "E3", "free": 1, "win": 0, "offer": 1, "Class": "Spam"},
    {"Email": "E4", "free": 0, "win": 1, "offer": 1, "Class": "Spam"},
    {"Email": "E5", "free": 0, "win": 0, "offer": 1, "Class": "NotSpam"},
    {"Email": "E6", "free": 0, "win": 1, "offer": 0, "Class": "NotSpam"},
    {"Email": "E7", "free": 1, "win": 0, "offer": 0, "Class": "NotSpam"},
]

df_train = pd.DataFrame(training_data)
features = ["free", "win", "offer"]
classes = ["Spam", "NotSpam"]

print("=== Training Dataset ===")
df_train


=== Training Dataset ===


,Email,free,win,offer,Class
0,E1,1,1,1,Spam
1,E2,1,1,0,Spam
2,E3,1,0,1,Spam
3,E4,0,1,1,Spam
4,E5,0,0,1,NotSpam
5,E6,0,1,0,NotSpam
6,E7,1,0,0,NotSpam


In [2]:
# Step 1: Calculate Class Priors
total_emails = len(df_train)
spam_data = df_train[df_train["Class"] == "Spam"]
not_spam_data = df_train[df_train["Class"] == "NotSpam"]

count_spam = len(spam_data)
count_not_spam = len(not_spam_data)

prior_spam = count_spam / total_emails
prior_not_spam = count_not_spam / total_emails

df_priors = pd.DataFrame({
    "Class": ["Spam", "NotSpam"],
    "Count": [count_spam, count_not_spam],
    "Prior Formula": [f"{count_spam}/{total_emails}", f"{count_not_spam}/{total_emails}"],
    "Prior Probability P(C)": [prior_spam, prior_not_spam]
})
print("=== Class Priors ===")
df_priors


=== Class Priors ===


,Class,Count,Prior Formula,Prior Probability P(C)
0,Spam,4,4/7,0.571429
1,NotSpam,3,3/7,0.428571


In [3]:
# Step 2: Calculate Feature Likelihoods
likelihood_list = []

for feature in features:
    # Spam: count of 1s and 0s
    c1_spam = spam_data[feature].sum()
    c0_spam = count_spam - c1_spam
    p1_spam = c1_spam / count_spam
    p0_spam = c0_spam / count_spam
    
    # NotSpam: count of 1s and 0s
    c1_not_spam = not_spam_data[feature].sum()
    c0_not_spam = count_not_spam - c1_not_spam
    p1_not_spam = c1_not_spam / count_not_spam
    p0_not_spam = c0_not_spam / count_not_spam
    
    likelihood_list.append({
        "Feature": feature,
        "P(x=1 | Spam)": f"{c1_spam}/{count_spam} = {p1_spam:.4f}",
        "P(x=0 | Spam)": f"{c0_spam}/{count_spam} = {p0_spam:.4f}",
        "P(x=1 | NotSpam)": f"{c1_not_spam}/{count_not_spam} = {p1_not_spam:.4f}",
        "P(x=0 | NotSpam)": f"{c0_not_spam}/{count_not_spam} = {p0_not_spam:.4f}"
    })

df_likelihoods = pd.DataFrame(likelihood_list)
print("=== Feature Likelihoods Table ===")
df_likelihoods


=== Feature Likelihoods Table ===


,Feature,P(x=1 | Spam),P(x=0 | Spam),P(x=1 | NotSpam),P(x=0 | NotSpam)
0,free,3/4 = 0.7500,1/4 = 0.2500,1/3 = 0.3333,2/3 = 0.6667
1,win,3/4 = 0.7500,1/4 = 0.2500,1/3 = 0.3333,2/3 = 0.6667
2,offer,3/4 = 0.7500,1/4 = 0.2500,1/3 = 0.3333,2/3 = 0.6667


In [4]:
# Step 3: Compute Posterior Scores for Email E8 = (0, 0, 0)
e8 = {"free": 0, "win": 0, "offer": 0}

# Likelihood product for Spam
# P(free=0|Spam) = 1/4
# P(win=0|Spam)  = 1/4
# P(offer=0|Spam)= 1/4
score_spam = prior_spam * (1/4) * (1/4) * (1/4)

# Likelihood product for NotSpam
# P(free=0|NotSpam) = 2/3
# P(win=0|NotSpam)  = 2/3
# P(offer=0|NotSpam)= 2/3
score_not_spam = prior_not_spam * (2/3) * (2/3) * (2/3)

# Normalized posterior probabilities
normalizer = score_spam + score_not_spam
prob_spam = score_spam / normalizer
prob_not_spam = score_not_spam / normalizer

print("=== Complete Computation for E8 = (0, 0, 0) ===")
print("1. Unnormalized Posterior for Spam:")
print(f"   Score(Spam) = P(Spam) * P(free=0|Spam) * P(win=0|Spam) * P(offer=0|Spam)")
print(f"               = (4/7) * (1/4) * (1/4) * (1/4)")
print(f"               = 4 / 448 = 1 / 112 = {score_spam:.8f}")

print("\n2. Unnormalized Posterior for NotSpam:")
print(f"   Score(NotSpam) = P(NotSpam) * P(free=0|NotSpam) * P(win=0|NotSpam) * P(offer=0|NotSpam)")
print(f"                  = (3/7) * (2/3) * (2/3) * (2/3)")
print(f"                  = 24 / 189 = 8 / 63 = {score_not_spam:.8f}")

print(f"\n3. Comparing Scores:")
print(f"   Score(NotSpam) = {score_not_spam:.6f} > Score(Spam) = {score_spam:.6f}")
print(f"   Normalized P(NotSpam | E8) = {prob_not_spam * 100:.2f}%")
print(f"   Normalized P(Spam | E8)    = {prob_spam * 100:.2f}%")

prediction = "NotSpam" if score_not_spam > score_spam else "Spam"
print(f"\n--> FINAL PREDICTION FOR E8: {prediction}")


=== Complete Computation for E8 = (0, 0, 0) ===
1. Unnormalized Posterior for Spam:
   Score(Spam) = P(Spam) * P(free=0|Spam) * P(win=0|Spam) * P(offer=0|Spam)
               = (4/7) * (1/4) * (1/4) * (1/4)
               = 4 / 448 = 1 / 112 = 0.00892857

2. Unnormalized Posterior for NotSpam:
   Score(NotSpam) = P(NotSpam) * P(free=0|NotSpam) * P(win=0|NotSpam) * P(offer=0|NotSpam)
                  = (3/7) * (2/3) * (2/3) * (2/3)
                  = 24 / 189 = 8 / 63 = 0.12698413

3. Comparing Scores:
   Score(NotSpam) = 0.126984 > Score(Spam) = 0.008929
   Normalized P(NotSpam | E8) = 93.43%
   Normalized P(Spam | E8)    = 6.57%

--> FINAL PREDICTION FOR E8: NotSpam


In [5]:
# Step 4: Verification with Laplace (Add-1) Smoothing
laplace_score_spam = prior_spam * ((1+1)/(4+2)) * ((1+1)/(4+2)) * ((1+1)/(4+2))
laplace_score_not_spam = prior_not_spam * ((2+1)/(3+2)) * ((2+1)/(3+2)) * ((2+1)/(3+2))

df_summary_methods = pd.DataFrame([
    {
        "Estimation Method": "Standard MLE (No Smoothing)",
        "Score(Spam)": f"1/112 ≈ {score_spam:.6f}",
        "Score(NotSpam)": f"8/63 ≈ {score_not_spam:.6f}",
        "Normalized P(NotSpam)": f"{prob_not_spam*100:.2f}%",
        "Predicted Class": "NotSpam"
    },
    {
        "Estimation Method": "Laplace Smoothing (Add-1)",
        "Score(Spam)": f"4/189 ≈ {laplace_score_spam:.6f}",
        "Score(NotSpam)": f"81/875 ≈ {laplace_score_not_spam:.6f}",
        "Normalized P(NotSpam)": f"{(laplace_score_not_spam/(laplace_score_spam+laplace_score_not_spam))*100:.2f}%",
        "Predicted Class": "NotSpam"
    }
])
print("=== Comparison of Estimation Methods ===")
df_summary_methods


=== Comparison of Estimation Methods ===


,Estimation Method,Score(Spam),Score(NotSpam),Normalized P(NotSpam),Predicted Class
0,Standard MLE (No Smoothing),1/112 ≈ 0.008929,8/63 ≈ 0.126984,93.43%,NotSpam
1,Laplace Smoothing (Add-1),4/189 ≈ 0.021164,81/875 ≈ 0.092571,81.39%,NotSpam


## Summary of Answers

1. **Computations [4 Marks]:**
   - $P(\text{Spam} \mid E8) \propto \frac{4}{7} \times \frac{1}{4} \times \frac{1}{4} \times \frac{1}{4} = \frac{1}{112} \approx \mathbf{0.008929}$
   - $P(\text{NotSpam} \mid E8) \propto \frac{3}{7} \times \frac{2}{3} \times \frac{2}{3} \times \frac{2}{3} = \frac{8}{63} \approx \mathbf{0.126984}$

2. **Prediction [2 Marks]:**
   - Since $\text{Score}(\text{NotSpam}) > \text{Score}(\text{Spam})$, the predicted class for email **E8** is **`NotSpam`**.
